In [ ]:
# Calculator Model Training and SHAP/FFA Workflow

**Purpose:** Train calculator models and run SHAP + Formal Feature Attribution (FFA) analysis  
**Updated:** January 27, 2026  
**Hardware:** Optimized for EC2 instances  
**Model Strategy:** Dual model implementation (Baseline + Extended) for all cohorts

# Calculator Model Training and SHAP/FFA Workflow

**Purpose:** Train calculator models and run SHAP + Formal Feature Attribution (FFA) analysis  
**Updated:** January 27, 2026  
**Hardware:** Optimized for EC2 instances  
**Model Strategy:** Dual model implementation (Baseline + Extended) for all cohorts

## Overview

This notebook provides an interactive workflow for:

1. **Training Calculator Models** - Train **two model variants** (Baseline and Extended) using the **Combined** cohort (single model for all patients)
2. **SHAP + FFA Analysis** - Generate causal factors and dashboard data using SHAP values and XGBoost rule extraction for both models
3. **Results Inspection** - View top causal factors, feature importance, and model performance
4. **Dashboard Deployment** - Deploy both models to the risk calculator dashboard with separate tabs

## Model Architecture

- **Dual Model Approach**: Two Combined models are trained for all cohorts (CHD, Cardiomyopathy, Myocarditis):
  - **Baseline Model** (`Combined_base`): Uses base calculator features only
  - **Extended Model** (`Combined_enhanced`): Uses base features + recommended additional features
- **Primary Diagnosis Feature**: `primary_etiology` is included to distinguish between etiologies
- **Feature Engineering**: Automatic derivation of combined variables (VAD, Ventilation, ECMO, donor ratios)

## Model Training Strategy

**Two Model Variants:**
1. **Baseline Model** (`Combined_base`): Base calculator features only (~104 features)
2. **Extended Model** (`Combined_enhanced`): Base features + recommended features (~120 features)

**For Each Model Variant, Three Model Types Trained:**
1. **CatBoost** - Gradient boosting with categorical feature support (Cox regression)
2. **XGBoost** - Extreme gradient boosting (Cox regression)
3. **XGBoost Random Forest** - XGBoost in Random Forest mode (Cox regression)

**Model Selection:**
- All three model types are trained on the same training data for each variant
- Performance is evaluated using C-index (Concordance Index)
- The model with the **highest C-index** is selected as the best model for each variant
- Both baseline and extended models are saved and deployed to the dashboard

**Dashboard Deployment:**
- **Baseline Model Tab**: Uses models from `Combined_base/` directory
- **Extended Model Tab**: Uses models from `Combined_enhanced/` directory
- Users can compare predictions from both models side-by-side

**Note:** This dual model approach allows users to choose between a simpler baseline model and a more comprehensive extended model with additional clinical features.

## Causal Analysis Strategy (SHAP + FFA)

**Important:** The causal analysis workflow uses a specific combination of models and applies rules to the **test set** for final causal analysis.

### Workflow Overview

```mermaid
graph TD
    A[Training Data] --> B[Temporal Split<br/>80/20]
    B --> C[Train Set<br/>txpl_year ≤ cutoff]
    B --> D[Test Set<br/>txpl_year > cutoff]
    
    C --> E[Train Models<br/>CatBoost, XGBoost, XGBoost RF]
    E --> F[Select Best Model<br/>by C-index, then AU-PRC]
    F --> G[Final Model<br/>Trained on Train Set]
    
    G --> H[Extract Rules<br/>from XGBoost JSON]
    D --> I[Compute SHAP Values<br/>on Test Set Only]
    
    H --> J[Apply Rules to Test Set<br/>Count Rule Firings]
    I --> K[Combine SHAP Values<br/>XGBoost + CatBoost if needed]
    
    J --> L[Calculate Rule Frequencies<br/>from Test Set]
    K --> M[SHAP Importance<br/>per Feature]
    
    L --> N[Causal Responsibility<br/>rule_freq × SHAP_importance]
    M --> N
    
    N --> O[Top K Causal Factors<br/>for Dashboard]
    
    style D fill:#e1f5ff
    style I fill:#e1f5ff
    style J fill:#e1f5ff
    style L fill:#e1f5ff
    style O fill:#c8e6c9
```

### Key Principles

**1. Test Set Application (Critical)**
- ✅ **Rules are extracted from the trained model** (trained on training set)
- ✅ **Rules are applied to the test set** (unseen data) for final causal analysis
- ✅ **SHAP values are computed on the test set only** (not training set)
- ✅ **Rule frequencies are counted from test set rule firings** (not from rule definitions)
- ✅ **Temporal split cutoff matches training** (dynamic 80/20 split, falls back to 2021)

**Why Test Set?**
- Ensures causal factors reflect model behavior on **unseen data**
- Prevents overfitting to training patterns
- Provides realistic causal responsibility scores
- Matches model evaluation methodology

### SHAP Values (Feature Importance)
- **Best XGBoost Model**: SHAP values are **always** computed from the best XGBoost model
- **Best CatBoost Model**: SHAP values are computed from the best CatBoost model **only if CatBoost is the best model**
- **Combination**: If CatBoost is best, SHAP values are combined with **auto-determined weights** based on C-index values
- **If XGBoost is best**: Only XGBoost SHAP values are used
- **Data Source**: SHAP values computed on **test set only** (`txpl_year > cutoff_year`)

### FFA Analysis (Rule Extraction)
- **Best XGBoost JSON Model**: Rules are **always** extracted from the best XGBoost JSON model
  - This is because XGBoost JSON structure is easier to parse for rule extraction
  - CatBoost JSON is not used (harder to parse due to categorical hashing)
- **Rule Source**: Rules extracted from model trained on **training set**
- **Rule Application**: Rules are **applied to test set instances** to count actual rule firings
- **Rule Filtering**: Rules are filtered using SHAP importance values (from step above)
- **Causal Responsibility**: Calculated as `(rule_frequency_from_test_set / total_rule_firings) × SHAP_importance`
  - `rule_frequency_from_test_set`: Count of how many times a rule fires on test set instances
  - `SHAP_importance`: Feature importance from SHAP values (computed on test set)

### Summary
**For causal analysis:**
1. ✅ SHAP values from **best XGBoost model** (always, computed on **test set**)
2. ✅ SHAP values from **best CatBoost model** (only if CatBoost is best, computed on **test set**)
3. ✅ Rules extracted from **best XGBoost JSON model** (trained on **training set**)
4. ✅ Rules applied to **test set** to count actual rule firings
5. ✅ FFA analysis combines **test set rule frequencies** + **test set SHAP values** to calculate causal responsibility

## Workflow Steps

- **Step 1:** Train Baseline Combined model (base calculator features)
- **Step 2:** Train Extended Combined model (base + recommended features)
- **Step 3:** Run SHAP/FFA analysis for Baseline model
- **Step 4:** Run SHAP/FFA analysis for Extended model
- **Step 5:** Inspect results and export dashboard data for both models
- **Step 6:** Deploy both models to dashboard (Baseline and Extended tabs)

## Expected Runtime

- **Baseline Model Training:** ~15-30 minutes
- **Extended Model Training:** ~15-30 minutes (with parallel processing)
- **SHAP/FFA Analysis (Baseline):** ~10-20 minutes
- **SHAP/FFA Analysis (Extended):** ~10-20 minutes
- **Total:** ~50-100 minutes on EC2 (can be parallelized)


## 1. Input Features Overview

### Required Input Variables for Risk Calculator

The model uses the following input features, with automatic feature engineering for derived variables:

#### Primary Diagnosis & History
- **Primary Diagnosis** (`primary_etiology`) - Congenital Heart Disease, Cardiomyopathy, Myocarditis, Other
- **Previous Cardiac Surgery** (`hxsurg`) - History of surgery (Yes/No)
- **Laterality Disorder** (`chd_lat`) - Composite variable (Yes/No)
  - Derived from: `chd_dex`, `chd_si`, `chd_heter`, `chd_iivc`, `chd_bivc`, `chd_lsvc`, `chd_raa`, `chd_avd`

#### Cardiac Support Devices (Combined Variables)
- **ECMO** (`ecmo_combined`) - ECMO at transplant OR listing
  - Derived from: `txecmo` OR `slecmo`
- **VAD** (`vad_combined`) - VAD at transplant OR listing
  - Derived from: `txvad` OR `slvad`
- **Mechanical Ventilation** (`vent_combined`) - Ventilation at transplant OR listing
  - Derived from: `txvent` OR `slvent` OR `ltxtrach` OR `hxtrach`

#### Demographics & Age
- **Age at Transplant** (`age_txpl`) - Years (priority over `age_listing`)
- **Age at Listing** (`age_listing`) - Years (fallback)

#### Renal Function
- **Dialysis History** (`hxdysdia` / `hxdysdia_bin`) - History of dialysis (ever)
- **eGFR at Transplant** (`egfr_tx`) - Calculated from height and creatinine
  - Formula: `egfr_tx = 0.413 × height_txpl / txcreat_r`
- **eGFR at Listing** (`egfr_listing`) - Calculated from height and creatinine

#### Liver Function
- **ALT at Transplant** (`txalt`) - U/L (priority over `lsalt`)
- **AST at Transplant** (`txast`) - U/L (priority over `lsast`)
- **Direct Bilirubin at Transplant** (`txbili_d_r`) - mg/dL (priority over `lsbili_d_r`)
- **Total Bilirubin at Transplant** (`txbili_t_r`) - mg/dL (priority over `lsbili_t_r`)

#### Nutrition
- **Serum Albumin at Transplant** (`txsa_r`) - g/dL (priority over `lssab_r`)
- **Total Protein at Transplant** (`txtp_r`) - g/dL (priority over `lstp_r`)

#### Immunology
- **cPRA at Transplant** (`txfcpra`) - Flow cytometry PRA % (priority over `lsfcpra`)
- **cPRA at Listing** (`lsfcpra`) - Flow cytometry PRA % (fallback)

#### Donor Characteristics
- **Donor Ischemic Time** (`donisch`) - Minutes (default: < 240 minutes if not provided)
- **Donor/Recipient Weight Ratio** (`donor_weight_ratio`) - Percentage
  - Formula: `(weight_donor / weight_txpl) × 100`
  - Model assumption: 70-200%
- **Donor/Recipient Size Ratio** (`donor_size_ratio`) - Percentage
  - Formula: `(height_donor / height_txpl) × 100`
  - Model assumption: 70-200%

### Additional Features

The model also includes:
- All CHD subtype variables (40+ subtypes, e.g., `chd_hlh`, `chd_lsvc`, `chd_si`, etc.)
- Additional lab values and clinical history variables
- Derived categorical variables (eGFR categories, high/low indicators)
- Donor characteristics and transplant details

### Feature Engineering

The following variables are automatically created during training and inference:
1. `ecmo_combined` - ECMO combined
2. `vad_combined` - VAD combined
3. `vent_combined` - Ventilation combined
4. `donor_weight_ratio` - Donor/recipient weight ratio
5. `donor_size_ratio` - Donor/recipient height ratio
6. `chd_lat` - Laterality disorder composite
7. `egfr_tx` - eGFR at transplant (if not provided, calculated from height/creatinine)
8. `egfr_listing` - eGFR at listing
9. `egfr_tx_cat` - eGFR category (severe/moderate/mild/normal)
10. `egfr_listing_cat` - eGFR category at listing
11. Additional derived variables (BMI, high/low indicators, etc.)

---

## 2. Setup and Configuration

Load required packages and configure paths.

In [ ]:
import sys
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# Add project paths
PROJECT_ROOT = Path().resolve().parent.parent.parent
CALCULATOR_DIR = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(CALCULATOR_DIR))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("=" * 80)
print("PHTS Calculator Workflow")
print("=" * 80)
print(f"Project root: {PROJECT_ROOT}")
print(f"Calculator directory: {CALCULATOR_DIR}")
print("=" * 80)

In [ ]:
# Timing helper for workflow steps (aligned with mermaid chart workflow)
import time
from contextlib import contextmanager

@contextmanager
def step_timer(step_name, sub_steps=None):
    """
    Context manager to time workflow steps with logging.
    
    Aligns with mermaid chart workflow:
    - Training: Temporal Split → Train Models → Select Best Model → Final Model
    - SHAP/FFA: Extract Rules → Compute SHAP → Apply Rules → Calculate Frequencies → Causal Responsibility
    
    Args:
        step_name: Main step name (e.g., "Step 1: Train Baseline Model")
        sub_steps: Optional list of sub-steps that align with mermaid chart nodes
    """
    start_time = time.time()
    start_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(start_time))
    print(f"\n{'=' * 80}")
    print(f"⏱️  START: {step_name}")
    print(f"   Started at: {start_str}")
    if sub_steps:
        print(f"   Sub-steps (per mermaid chart):")
        for i, sub_step in enumerate(sub_steps, 1):
            print(f"     {i}. {sub_step}")
    print(f"{'=' * 80}")
    logger.info(f"START: {step_name} at {start_str}")
    if sub_steps:
        logger.info(f"Sub-steps: {', '.join(sub_steps)}")
    
    try:
        yield
    finally:
        end_time = time.time()
        end_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(end_time))
        duration = end_time - start_time
        duration_min = duration / 60
        duration_sec = duration % 60
        
        print(f"\n{'=' * 80}")
        print(f"✅ COMPLETE: {step_name}")
        print(f"   Started: {start_str}")
        print(f"   Finished: {end_str}")
        print(f"   Duration: {duration_min:.1f} minutes ({duration:.0f} seconds)")
        print(f"{'=' * 80}")
        logger.info(f"COMPLETE: {step_name} - Duration: {duration_min:.1f} minutes ({duration:.0f} seconds)")

print("✓ Timing helper loaded (aligned with mermaid chart workflow)")

In [ ]:
# Configuration
DEBUG_MODE = False  # Set to True for quick testing (fewer splits)

# Model Strategy: Single Combined model for all cohorts
# Note: The training script will always train Combined model regardless of --cohort argument
COHORT = "Combined"  # Single model approach - Combined model for all cohorts

# SHAP/FFA configuration
TOP_K = 10  # Number of top causal factors to extract
# Note: Weights are automatically determined from best model C-index values
# Set to None to use auto-determination, or override manually if needed
WEIGHT_CATBOOST = None  # Auto-determined from best model (None = auto)
WEIGHT_XGBOOST = None   # Auto-determined from best model (None = auto)

print(f"\nConfiguration:")
print(f"  DEBUG_MODE: {DEBUG_MODE}")
print(f"  Model Strategy: Single Combined model (for all cohorts)")
print(f"  Cohort: {COHORT}")
print(f"  Top K factors: {TOP_K}")
print(f"  SHAP Weights: Auto-determined from best model C-index values")
print(f"    (CatBoost weight: {'Auto' if WEIGHT_CATBOOST is None else WEIGHT_CATBOOST})")
print(f"    (XGBoost weight: {'Auto' if WEIGHT_XGBOOST is None else WEIGHT_XGBOOST})")
print(f"\nNote: The model includes primary_etiology to distinguish between:")
print(f"  - Congenital Heart Disease")
print(f"  - Cardiomyopathy")
print(f"  - Myocarditis")
print(f"  - Other")

In [ ]:
# Check dependencies
print("\nChecking dependencies...")

try:
    import numpy as np
    import pandas as pd
    from catboost import CatBoostRegressor
    import xgboost as xgb
    import shap
    print("✓ All required packages are installed")
    print(f"  NumPy: {np.__version__}")
    print(f"  Pandas: {pd.__version__}")
    print(f"  XGBoost: {xgb.__version__}")
    print(f"  SHAP: {shap.__version__}")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("  Please install: pip install numpy pandas catboost xgboost shap")

In [ ]:
# Check data availability
print("\nChecking data availability...")

data_file = PROJECT_ROOT / "graft-loss" / "data" / "phts_txpl_ml.sas7bdat"
if data_file.exists():
    size_mb = data_file.stat().st_size / (1024 * 1024)
    print(f"✓ Data file found: {data_file}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"⚠ Data file not found: {data_file}")
    print("  You may need to download the data file first")

# Check calculator directory structure
outputs_dir = CALCULATOR_DIR / "outputs"
if outputs_dir.exists():
    print(f"✓ Outputs directory exists: {outputs_dir}")
else:
    print(f"✓ Creating outputs directory: {outputs_dir}")
    outputs_dir.mkdir(parents=True, exist_ok=True)

## 3. Train Calculator Models

Train **all three model types** (CatBoost, XGBoost, and XGBoost RF) using the **Combined** cohort.

**Training Process:**
1. **Data Split**: Temporal 80/20 split (train on earlier years, test on later years)
2. **Model Training**: All three models are trained on the same training data:
   - CatBoost (Cox regression)
   - XGBoost (Cox regression)
   - XGBoost Random Forest (Cox regression)
3. **Model Evaluation**: C-index is calculated for each model on the test set
4. **Model Selection**: The model with the highest C-index is selected as the best model

**Note:** The training script enforces a single Combined model strategy. Even if you specify a different cohort, it will train the Combined model for all patients.

In [ ]:
# Import training function
from train_python_models import train_models_for_cohort
import os
import multiprocessing

# Determine number of parallel jobs (use all available CPUs minus 1 for safety)
n_parallel_jobs = max(1, multiprocessing.cpu_count() - 1)

print(f"\n{'=' * 80}")
print("Training Baseline Calculator Model (Base Features Only)")
print(f"{'=' * 80}")
print(f"\nConfiguration:")
print(f"  Cohort: {COHORT}")
print(f"  Feature Set: Baseline (base calculator features only)")
print(f"  Parallel Jobs: {n_parallel_jobs} (using {multiprocessing.cpu_count()} CPUs)")
print(f"  MC-CV Splits: 25")
print(f"  Training Proportion: 80%")
print(f"\nTraining Process:")
print(f"  1. Monte Carlo Cross-Validation (25 splits)")
print(f"  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)")
print(f"  3. Model Selection: Best model by C-index, then AU-PRC")
print(f"  4. Final Model: Best model trained on full temporal split")
print("-" * 80)
print("\nModel Features:")
print("  - primary_etiology feature to distinguish etiologies")
print("  - All derived variables (vad_combined, vent_combined, donor ratios, chd_lat)")
print("  - All required input features from risk calculator")
print("-" * 80)

print(f"\nTraining Baseline Combined model (for all cohorts)...")
print(f"  Output directory: outputs/models/Combined_base/")

try:
    import time
    start_time = time.time()
    
    train_models_for_cohort(
        cohort=COHORT,
        n_mc_splits=25,
        train_prop=0.8,
        n_jobs=n_parallel_jobs,  # Use parallel processing
        include_recommended_features=False  # Baseline model uses base features only
    )
    
    elapsed_time = time.time() - start_time
    print(f"\n✓ Baseline Combined model training complete!")
    print(f"  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)")
    print(f"  Output saved to: outputs/models/Combined_base/")
    print(f"\nAll three models have been trained:")
    print(f"  - CatBoost")
    print(f"  - XGBoost")
    print(f"  - XGBoost Random Forest")
    print(f"\nThe best model (highest C-index) has been selected and saved.")
    print(f"\nThe model is now ready for:")
    print(f"  - Risk prediction for all cohorts (CHD, Cardiomyopathy, Myocarditis)")
    print(f"  - SHAP/FFA analysis to extract causal factors")
except Exception as e:
    print(f"\n✗ Error training Baseline Combined model: {e}")
    import traceback
    traceback.print_exc()

print(f"\n{'=' * 80}")
print("Baseline Model Training Complete!")
print(f"{'=' * 80}")

In [ ]:
# Check training results
import json

print("\nTraining Results Summary:")
print("-" * 80)

best_model_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "best_model.txt"
if best_model_file.exists():
    print(f"\n{COHORT} Model (for all cohorts):")
    with open(best_model_file, 'r') as f:
        content = f.read()
        print(content)
        
        # Extract model performance info
        lines = content.split('\n')
        model_performances = {}
        for line in lines:
            if 'CatBoost C-index:' in line:
                model_performances['CatBoost'] = line.split(':')[1].strip()
            elif 'XGBoost C-index:' in line:
                model_performances['XGBoost'] = line.split(':')[1].strip()
            elif 'XGBoost RF C-index:' in line:
                model_performances['XGBoost RF'] = line.split(':')[1].strip()
        
        if model_performances:
            print(f"\n  Model Performance Comparison:")
            for model_name, c_index in sorted(model_performances.items(), 
                                               key=lambda x: float(x[1]), 
                                               reverse=True):
                print(f"    {model_name:20s} C-index: {c_index}")
else:
    print(f"\n⚠ {COHORT}: Best model file not found")

# List model files
models_dir = CALCULATOR_DIR / "outputs" / "models" / COHORT
if models_dir.exists():
    # Check for all three model types
    catboost_file = models_dir / "catboost_model.cbm"
    xgboost_file = models_dir / "xgboost_model.ubj"
    xgboost_rf_file = models_dir / "xgboost_rf_model.ubj"
    
    print(f"\n  Trained Models:")
    if catboost_file.exists():
        size_mb = catboost_file.stat().st_size / (1024 * 1024)
        print(f"    ✓ CatBoost: {catboost_file.name} ({size_mb:.2f} MB)")
    else:
        print(f"    ○ CatBoost: Not found")
    
    if xgboost_file.exists():
        size_mb = xgboost_file.stat().st_size / (1024 * 1024)
        print(f"    ✓ XGBoost: {xgboost_file.name} ({size_mb:.2f} MB)")
    else:
        print(f"    ○ XGBoost: Not found")
    
    if xgboost_rf_file.exists():
        size_mb = xgboost_rf_file.stat().st_size / (1024 * 1024)
        print(f"    ✓ XGBoost RF: {xgboost_rf_file.name} ({size_mb:.2f} MB)")
    else:
        print(f"    ○ XGBoost RF: Not found")
    
    # Check feature count
    feature_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "feature_names.json"
    if feature_file.exists():
        with open(feature_file, 'r') as f:
            features = json.load(f)
            print(f"\n  Feature Summary:")
            print(f"    Total features: {len(features)}")
            print(f"    ✓ primary_etiology: {'primary_etiology' in features or any('primary_etiology' in f for f in features)}")
            print(f"    ✓ vad_combined: {'vad_combined' in features}")
            print(f"    ✓ vent_combined: {'vent_combined' in features}")
            print(f"    ✓ ecmo_combined: {'ecmo_combined' in features}")
            print(f"    ✓ donor_weight_ratio: {'donor_weight_ratio' in features}")
            print(f"    ✓ donor_size_ratio: {'donor_size_ratio' in features}")
            print(f"    ✓ chd_lat: {'chd_lat' in features}")

## 3b. Train Enhanced Model (with Recommended Features)

Train the enhanced calculator model with recommended additional features (BNP, CRP, sec_dx/ter_dx, lipid panel, etc.) to compare performance with the base model.

**Enhanced Features Include:**
- BNP (Brain Natriuretic Peptide): `txbnp`, `txpbnp_r`, `lbnp`, `lspbnp_r`
- CRP (C-Reactive Protein): `txcrp_r`, `lcrp_r`
- Secondary/Tertiary diagnoses: `sec_dx`, `ter_dx`
- Pre-albumin at listing: `lspalb_r`
- Lipid panel: `txchol_r`, `txtg_r`, `txldl_r`, `txhdl_r`, `txvldl_r`
- Oxygen saturation: `txbaosat`, `txsvcsat`, `lsbaosat`, `lssvcsat`

**Note:** This model will be saved to `outputs/models/Combined_enhanced/` for comparison with the base model.

In [ ]:
# Import training function
from train_python_models import train_models_for_cohort
import os
import multiprocessing

# Determine number of parallel jobs (use all available CPUs minus 1 for safety)
n_parallel_jobs = max(1, multiprocessing.cpu_count() - 1)

print("=" * 80)
print("Training Enhanced Calculator Model (with Recommended Features)")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  Cohort: {COHORT}")
print(f"  Feature Set: Enhanced (base + recommended features)")
print(f"  Parallel Jobs: {n_parallel_jobs} (using {multiprocessing.cpu_count()} CPUs)")
print(f"  MC-CV Splits: 25")
print(f"  Training Proportion: 80%")
print(f"\nEnhanced Features:")
print(f"  - BNP values (txbnp, txpbnp_r, lbnp, lspbnp_r)")
print(f"  - CRP (txcrp_r, lcrp_r)")
print(f"  - Secondary/Tertiary diagnoses (sec_dx, ter_dx)")
print(f"  - Pre-albumin at listing (lspalb_r)")
print(f"  - Lipid panel (txchol_r, txtg_r, txldl_r, txhdl_r, txvldl_r)")
print(f"  - Oxygen saturation (txbaosat, txsvcsat, lsbaosat, lssvcsat)")

print(f"\nTraining Process:")
print(f"  1. Monte Carlo Cross-Validation (25 splits)")
print(f"  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)")
print(f"  3. Model Selection: Best model by C-index, then AU-PRC")
print(f"  4. Final Model: Best model trained on full temporal split")

print(f"\nTraining Enhanced Combined model (for all cohorts)...")
print(f"  Output directory: outputs/models/Combined_enhanced/")

try:
    import time
    start_time = time.time()
    
    train_models_for_cohort(
        cohort=COHORT,
        n_mc_splits=25,
        train_prop=0.8,
        n_jobs=n_parallel_jobs,  # Use parallel processing
        include_recommended_features=True  # Enable enhanced features
    )
    
    elapsed_time = time.time() - start_time
    print(f"\n✓ Enhanced Combined model training complete!")
    print(f"  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)")
    print(f"  Output saved to: outputs/models/Combined_enhanced/")
    
except Exception as e:
    print(f"\n✗ Error training Enhanced Combined model: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)
print("Enhanced Model Training Complete!")
print("=" * 80)

### 4b. SHAP/FFA Analysis for Extended Model

Run SHAP/FFA analysis for the extended model to generate causal factors and dashboard data for the Extended Model tab.

**Note:** The workflow will automatically detect the enhanced model directory (`Combined_enhanced`) and use those models for analysis.

In [ ]:
# Run SHAP/FFA workflow for Extended model
import subprocess

print(f"\n{'=' * 80}")
print("Running SHAP + FFA Analysis for Extended Model")
print(f"{'=' * 80}")
print(f"Analyzing Extended Combined model (for all cohorts)...")
print("-" * 80)
print(f"\nCausal Analysis Process:")
print(f"  1. Check best model (from best_model.txt in Combined_enhanced/)")
print(f"  2. Compute SHAP values:")
print(f"     - If XGBoost is best: XGBoost SHAP only")
print(f"     - If CatBoost is best: Combined SHAP (CatBoost + XGBoost)")
print(f"     - Weights auto-determined from C-index values")
print(f"  3. Extract rules from best XGBoost JSON model (if FFA available)")
print(f"  4. Filter rules using SHAP importance")
print(f"  5. Calculate causal responsibility (if FFA available)")
print(f"  6. Generate top {TOP_K} causal factors")
print("-" * 80)
print(f"\nModel Variant: Extended (base + recommended features)")
print(f"Output: outputs/shap_ffa/{COHORT}/ (dashboard data for Extended Model tab)")
print("-" * 80)
print(f"\nNote: The workflow will automatically detect and use models from Combined_enhanced/")
print(f"      directory. Make sure Extended model training (Step 3b) completed successfully.")
print("-" * 80)

# Build command - same as baseline but will use enhanced models
cmd = [
    sys.executable,
    str(CALCULATOR_DIR / "run_shap_ffa_workflow.py"),
    "--cohort", COHORT,
    "--top-k", str(TOP_K)
]

# Only add weight arguments if manually specified (otherwise auto-determined)
if WEIGHT_CATBOOST is not None:
    cmd.extend(["--weight-catboost", str(WEIGHT_CATBOOST)])
if WEIGHT_XGBOOST is not None:
    cmd.extend(["--weight-xgboost", str(WEIGHT_XGBOOST)])

try:
    result = subprocess.run(
        cmd,
        cwd=str(CALCULATOR_DIR),
        capture_output=False,  # Show output in real-time
        text=True
    )
    
    if result.returncode == 0:
        print(f"\n✓ Extended model SHAP/FFA analysis complete!")
        print(f"\nResults include:")
        print(f"  - Top {TOP_K} causal factors with causal responsibility scores")
        print(f"  - Feature importance rankings (SHAP-based)")
        print(f"  - Rule-based FFA analysis results")
        print(f"  - Dashboard data for Extended Model tab")
        print(f"\nNote: Causal factors are calculated using:")
        print(f"  - Rules extracted from best XGBoost JSON model (Combined_enhanced/)")
        print(f"  - SHAP importance from best models (XGBoost, and CatBoost if CatBoost is best)")
        print(f"  - Formula: causal_responsibility = (rule_frequency / total_rules) × SHAP_importance")
    else:
        print(f"\n⚠ SHAP/FFA exited with code: {result.returncode}")
except Exception as e:
    print(f"\n✗ Error running SHAP/FFA: {e}")
    logger.error(f"Error running SHAP/FFA", exc_info=True)

print(f"\n{'=' * 80}")
print("Extended Model SHAP/FFA analysis complete!")
print(f"{'=' * 80}")

## 4. Run SHAP + FFA Analysis

Generate SHAP values and extract causal factors using Formal Feature Attribution for the Combined model.

### Causal Analysis Workflow - Explicit Model Usage

**Step 1: SHAP Value Computation**

The workflow checks which model is best (from `best_model.txt`):

- **If XGBoost is best model:**
  - ✅ Computes SHAP values from **best XGBoost model** only
  - Uses simplified pipeline (XGBoost SHAP only)
  - No weights needed (single model)

- **If CatBoost is best model:**
  - ✅ Computes SHAP values from **best CatBoost model**
  - ✅ Computes SHAP values from **best XGBoost model**
  - ✅ **Automatically determines weights** based on C-index values:
    - Weights are calculated from relative C-index performance
    - CatBoost (best model) gets higher weight
    - XGBoost gets lower weight
    - Weights normalized to sum to 1.0
  - Uses combined pipeline (CatBoost + XGBoost SHAP)

**Step 2: FFA Rule Extraction**

- ✅ **Always uses**: Best XGBoost JSON model for rule extraction
  - Rules are extracted from XGBoost JSON structure (`*_final_model_xgboost.json`)
  - CatBoost JSON is **never** used for rule extraction (harder to parse due to categorical hashing)
  - Even if CatBoost is the best model, rules come from XGBoost JSON

**Step 3: Rule Filtering & Causal Responsibility**

- Rules are filtered using SHAP importance values (from Step 1)
- Causal responsibility is calculated as:
  ```
  causal_responsibility = (rule_frequency / total_rules) × SHAP_importance
  ```
- Where `SHAP_importance` comes from:
  - XGBoost SHAP only (if XGBoost is best), OR
  - Combined CatBoost + XGBoost SHAP (if CatBoost is best)

**Note:** FFA analysis is **REQUIRED**. The workflow requires `ffa_analysis/` directory with:
- `ffa_utils.py` (with `load_model_json`, `extract_feature_mappings`)
- `xgboost_axp_explainer.py` (with `XGBoostSymbolicExplainer`, `PathConfig`)

These modules have been restored from git history and are now available in the repository.

**Summary - Explicit Model Usage:**

| Component | Model Used | Condition |
|-----------|-----------|-----------|
| **SHAP Values** | Best XGBoost | Always |
| **SHAP Values** | Best CatBoost | Only if CatBoost is best model |
| **Rule Extraction** | Best XGBoost JSON | Always (regardless of which model is best) |
| **FFA Analysis** | Best XGBoost JSON + SHAP | Always |

**Key Point:** Even if CatBoost is the best model, the FFA analysis still uses the **XGBoost JSON model** for rule extraction, but filters those rules using **combined SHAP values** (CatBoost + XGBoost).

## 3b. Train Enhanced Model (with Recommended Features)

Train the enhanced calculator model with recommended additional features (BNP, CRP, sec_dx/ter_dx, lipid panel, etc.) to compare performance with the base model.

**Enhanced Features Include:**
- BNP (Brain Natriuretic Peptide): `txbnp`, `txpbnp_r`, `lbnp`, `lspbnp_r`
- CRP (C-Reactive Protein): `txcrp_r`, `lcrp_r`
- Secondary/Tertiary diagnoses: `sec_dx`, `ter_dx`
- Pre-albumin at listing: `lspalb_r`
- Lipid panel: `txchol_r`, `txtg_r`, `txldl_r`, `txhdl_r`, `txvldl_r`
- Oxygen saturation: `txbaosat`, `txsvcsat`, `lsbaosat`, `lssvcsat`

**Note:** This model will be saved to `outputs/models/Combined_enhanced/` for comparison with the base model.

In [ ]:
# Import training function
from train_python_models import train_models_for_cohort
import os

# Determine number of parallel jobs (use all available CPUs minus 1 for safety)
import multiprocessing
n_parallel_jobs = max(1, multiprocessing.cpu_count() - 1)

print("=" * 80)
print("Training Enhanced Calculator Model (with Recommended Features)")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  Cohort: {COHORT}")
print(f"  Feature Set: Enhanced (base + recommended features)")
print(f"  Parallel Jobs: {n_parallel_jobs} (using {multiprocessing.cpu_count()} CPUs)")
print(f"  MC-CV Splits: 25")
print(f"  Training Proportion: 80%")
print(f"\nEnhanced Features:")
print(f"  - BNP values (txbnp, txpbnp_r, lbnp, lspbnp_r)")
print(f"  - CRP (txcrp_r, lcrp_r)")
print(f"  - Secondary/Tertiary diagnoses (sec_dx, ter_dx)")
print(f"  - Pre-albumin at listing (lspalb_r)")
print(f"  - Lipid panel (txchol_r, txtg_r, txldl_r, txhdl_r, txvldl_r)")
print(f"  - Oxygen saturation (txbaosat, txsvcsat, lsbaosat, lssvcsat)")

print(f"\nTraining Process:")
print(f"  1. Monte Carlo Cross-Validation (25 splits)")
print(f"  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)")
print(f"  3. Model Selection: Best model by C-index, then AU-PRC")
print(f"  4. Final Model: Best model trained on full temporal split")

print(f"\nTraining Enhanced Combined model (for all cohorts)...")
print(f"  Output directory: outputs/models/Combined_enhanced/")

try:
    import time
    start_time = time.time()
    
    train_models_for_cohort(
        cohort=COHORT,
        n_mc_splits=25,
        train_prop=0.8,
        n_jobs=n_parallel_jobs,  # Use parallel processing
        include_recommended_features=True  # Enable enhanced features
    )
    
    elapsed_time = time.time() - start_time
    print(f"\n✓ Enhanced Combined model training complete!")
    print(f"  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)")
    print(f"  Output saved to: outputs/models/Combined_enhanced/")
    
except Exception as e:
    print(f"\n✗ Error training Enhanced Combined model: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)
print("Enhanced Model Training Complete!")
print("=" * 80)

In [ ]:
# Run SHAP/FFA workflow for Combined model
import subprocess

print(f"\n{'=' * 80}")
print("Running SHAP + FFA Analysis")
print(f"{'=' * 80}")
print(f"Analyzing Combined model (for all cohorts)...")
print("-" * 80)
print(f"\nCausal Analysis Process:")
print(f"  1. Check best model (from best_model.txt)")
print(f"  2. Compute SHAP values:")
print(f"     - If XGBoost is best: XGBoost SHAP only")
print(f"     - If CatBoost is best: Combined SHAP (CatBoost + XGBoost)")
print(f"     - Weights auto-determined from C-index values")
print(f"  3. Extract rules from best XGBoost JSON model (if FFA available)")
print(f"  4. Filter rules using SHAP importance")
print(f"  5. Calculate causal responsibility (if FFA available)")
print(f"  6. Generate top {TOP_K} causal factors")
print("-" * 80)
print(f"\nNote: If FFA modules are not available:")
print(f"  - Workflow continues with SHAP analysis only")
print(f"  - Uses SHAP importance instead of causal responsibility")
print(f"  - Dashboard will show SHAP-based rankings")
print("-" * 80)

# Build command - only include weight args if manually specified
cmd = [
    sys.executable,
    str(CALCULATOR_DIR / "run_shap_ffa_workflow.py"),
    "--cohort", COHORT,
    "--top-k", str(TOP_K)
]

# Only add weight arguments if manually specified (otherwise auto-determined)
if WEIGHT_CATBOOST is not None:
    cmd.extend(["--weight-catboost", str(WEIGHT_CATBOOST)])
if WEIGHT_XGBOOST is not None:
    cmd.extend(["--weight-xgboost", str(WEIGHT_XGBOOST)])

try:
    result = subprocess.run(
        cmd,
        cwd=str(CALCULATOR_DIR),
        capture_output=False,  # Show output in real-time
        text=True
    )
    
    if result.returncode == 0:
        print(f"\n✓ Combined model SHAP/FFA analysis complete!")
        print(f"\nResults include:")
        print(f"  - Top {TOP_K} causal factors with causal responsibility scores")
        print(f"  - Feature importance rankings (SHAP-based)")
        print(f"  - Rule-based FFA analysis results")
        print(f"  - Dashboard data for risk calculator")
        print(f"\nNote: Causal factors are calculated using:")
        print(f"  - Rules extracted from best XGBoost JSON model")
        print(f"  - SHAP importance from best models (XGBoost, and CatBoost if CatBoost is best)")
        print(f"  - Formula: causal_responsibility = (rule_frequency / total_rules) × SHAP_importance")
    else:
        print(f"\n⚠ SHAP/FFA exited with code: {result.returncode}")
except Exception as e:
    print(f"\n✗ Error running SHAP/FFA: {e}")
    logger.error(f"Error running SHAP/FFA", exc_info=True)

print(f"\n{'=' * 80}")
print("SHAP/FFA analysis complete!")
print(f"{'=' * 80}")

## 5. Inspect Results

View top causal factors, feature importance, and dashboard data for the Combined model.

In [ ]:
# Load and display dashboard data
import json
import pandas as pd

print("\n" + "=" * 80)
print("Results Summary - Combined Model")
print("=" * 80)

dashboard_data_file = (
    CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / "dashboard_data.json"
)

if dashboard_data_file.exists():
    print(f"\n{COHORT} Model - Top {TOP_K} Causal Factors:")
    print("-" * 80)
    
    with open(dashboard_data_file, 'r') as f:
        dashboard_data = json.load(f)
    
    top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
    
    if top_factors:
        for idx, factor in enumerate(top_factors, 1):
            importance = factor.get('causal_responsibility', 
                                 factor.get('importance', 
                                           factor.get('combined_importance_norm', 0)))
            print(f"{idx:2d}. {factor['feature']:40s} "
                  f"(Importance: {importance:.4f})")
    else:
        print("  (No causal factors available)")
    
    # Display summary statistics
    if 'summary' in dashboard_data:
        print(f"\n  Summary Statistics:")
        summary = dashboard_data['summary']
        for key, value in summary.items():
            print(f"    {key}: {value}")
    
    # Check for key features in top factors
    print(f"\n  Key Features Check:")
    top_feature_names = [f['feature'] for f in top_factors]
    key_features = {
        'primary_etiology': any('primary_etiology' in f for f in top_feature_names),
        'vad_combined': 'vad_combined' in top_feature_names,
        'vent_combined': 'vent_combined' in top_feature_names,
        'ecmo_combined': 'ecmo_combined' in top_feature_names,
        'donor_weight_ratio': 'donor_weight_ratio' in top_feature_names,
        'donor_size_ratio': 'donor_size_ratio' in top_feature_names,
        'chd_lat': 'chd_lat' in top_feature_names,
        'egfr_tx': 'egfr_tx' in top_feature_names or any('egfr' in f for f in top_feature_names),
        'txfcpra': 'txfcpra' in top_feature_names,
        'hxsurg': 'hxsurg' in top_feature_names
    }
    for feature, present in key_features.items():
        status = "✓" if present else "○"
        print(f"    {status} {feature}")
else:
    print(f"\n⚠ Dashboard data not found")
    print(f"  Expected: {dashboard_data_file}")
    print("  Run SHAP/FFA analysis first (Section 4)")

In [ ]:
# Load and display feature importance
print("\n" + "=" * 80)
print("Feature Importance Rankings - Combined Model")
print("=" * 80)

importance_files = list(
    (CALCULATOR_DIR / "outputs" / "models" / COHORT).glob("importance_*.csv")
)

if importance_files:
    print(f"\n{COHORT} Model - Feature Importance:")
    print("-" * 80)
    
    for imp_file in sorted(importance_files):
        model_name = imp_file.stem.replace(f"importance_{COHORT}_", "")
        print(f"\n  {model_name}:")
        df = pd.read_csv(imp_file)
        print(f"    Total features: {len(df)}")
        print(f"    Top 10 features:")
        top10 = df.nlargest(10, 'importance')
        for idx, row in top10.iterrows():
            print(f"      {row['feature']:40s} {row['importance']:.4f}")
        
        # Check for key features
        feature_list = df['feature'].tolist()
        print(f"\n    Key Features Status:")
        key_features = {
            'primary_etiology': any('primary_etiology' in f for f in feature_list),
            'vad_combined': 'vad_combined' in feature_list,
            'vent_combined': 'vent_combined' in feature_list,
            'ecmo_combined': 'ecmo_combined' in feature_list,
            'donor_weight_ratio': 'donor_weight_ratio' in feature_list,
            'donor_size_ratio': 'donor_size_ratio' in feature_list,
            'chd_lat': 'chd_lat' in feature_list,
            'egfr_tx': 'egfr_tx' in feature_list,
            'txfcpra': 'txfcpra' in feature_list,
            'hxsurg': 'hxsurg' in feature_list
        }
        for feature, present in key_features.items():
            status = "✓" if present else "○"
            if present:
                rank = df[df['feature'] == feature].index[0] + 1 if feature in feature_list else "N/A"
                print(f"      {status} {feature:25s} (Rank: {rank})")
            else:
                print(f"      {status} {feature:25s} (Not found)")
else:
    print(f"\n⚠ No feature importance files found")
    print("  Train models first (Section 3)")

## 6. Visualizations (Optional)

Create visualizations of results for the Combined model.

In [ ]:
# Plot top causal factors (if matplotlib is available)
try:
    import matplotlib.pyplot as plt
    
    dashboard_data_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / "dashboard_data.json"
    )
    
    if dashboard_data_file.exists():
        with open(dashboard_data_file, 'r') as f:
            dashboard_data = json.load(f)
        
        top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
        
        if top_factors:
            # Extract data for plotting
            features = [f['feature'] for f in top_factors]
            importance = [f.get('causal_responsibility', 
                              f.get('importance', 
                                   f.get('combined_importance_norm', 0))) 
                        for f in top_factors]
            
            # Create plot
            plt.figure(figsize=(10, max(6, len(features) * 0.4)))
            plt.barh(range(len(features)), importance)
            plt.yticks(range(len(features)), features)
            plt.xlabel('Causal Responsibility / Importance')
            plt.title(f'Top {TOP_K} Causal Factors - {COHORT} Model (All Cohorts)')
            plt.gca().invert_yaxis()  # Top factor at top
            plt.tight_layout()
            
            # Save plot
            plot_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / f"top_{TOP_K}_factors.png"
            plt.savefig(plot_file, dpi=150, bbox_inches='tight')
            print(f"\n✓ Saved plot: {plot_file}")
            
            plt.show()
        else:
            print("\n⚠ No causal factors available for plotting")
    else:
        print(f"\n⚠ Dashboard data not found: {dashboard_data_file}")
        print("  Run SHAP/FFA analysis first (Section 4)")
            
except ImportError:
    print("\n⚠ Matplotlib not available. Skipping visualizations.")
    print("  Install with: pip install matplotlib")

## 7. Export Summary

Create a summary JSON file with all results for the Combined model.

In [ ]:
# Create workflow summary
from datetime import datetime

summary = {
    "workflow": "Calculator Model Training + SHAP/FFA Analysis",
    "model_strategy": "Single Combined model for all cohorts",
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "cohort": COHORT,
        "top_k": TOP_K,
        "weight_catboost": WEIGHT_CATBOOST,
        "weight_xgboost": WEIGHT_XGBOOST,
        "debug_mode": DEBUG_MODE
    },
    "model": {}
}

# Best model
best_model_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "best_model.txt"
if best_model_file.exists():
    with open(best_model_file, 'r') as f:
        content = f.read()
        lines = content.split('\n')
        for line in lines:
            if line.startswith("Best Model:"):
                summary["model"]["best_model"] = line.replace("Best Model: ", "").strip()
            elif line.startswith("C-index:"):
                try:
                    summary["model"]["c_index"] = float(line.replace("C-index: ", "").strip())
                except:
                    pass

# Dashboard data
dashboard_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / "dashboard_data.json"
if dashboard_file.exists():
    with open(dashboard_file, 'r') as f:
        dashboard_data = json.load(f)
        summary["model"]["top_factors_count"] = len(dashboard_data.get('top_causal_factors', []))
        if dashboard_data.get('top_causal_factors'):
            summary["model"]["top_factor"] = dashboard_data['top_causal_factors'][0]['feature']
            summary["model"]["top_factor_importance"] = dashboard_data['top_causal_factors'][0].get(
                'causal_responsibility', 
                dashboard_data['top_causal_factors'][0].get('importance', 0)
            )
        
        # List top 5 factors
        top5 = dashboard_data.get('top_causal_factors', [])[:5]
        summary["model"]["top_5_factors"] = [
            {
                "feature": f['feature'],
                "importance": f.get('causal_responsibility', 
                                  f.get('importance', 
                                       f.get('combined_importance_norm', 0)))
            }
            for f in top5
        ]

# Feature count
feature_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "feature_names.json"
if feature_file.exists():
    with open(feature_file, 'r') as f:
        features = json.load(f)
        summary["model"]["total_features"] = len(features)
        summary["model"]["key_features"] = {
            "primary_etiology": 'primary_etiology' in features or any('primary_etiology' in f for f in features),
            "vad_combined": 'vad_combined' in features,
            "vent_combined": 'vent_combined' in features,
            "ecmo_combined": 'ecmo_combined' in features,
            "donor_weight_ratio": 'donor_weight_ratio' in features,
            "donor_size_ratio": 'donor_size_ratio' in features,
            "chd_lat": 'chd_lat' in features,
            "egfr_tx": 'egfr_tx' in features,
            "txfcpra": 'txfcpra' in features,
            "hxsurg": 'hxsurg' in features
        }

# Save summary
summary_file = CALCULATOR_DIR / "outputs" / "workflow_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Workflow summary saved to: {summary_file}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

## 8. Feature Validation

Validate that all required input features are present in the trained model.

---

## 9. Deploy Risk Calculator (Lambda + S3)

Deploy the trained models and dashboard to AWS Lambda and S3 for production use.

### Deployment Overview

**Components:**
- **AWS Lambda**: Container-based function with models baked in
- **API Gateway**: REST API endpoints for risk calculation
- **S3**: Static HTML dashboard hosting

**Prerequisites:**
- AWS CLI configured with appropriate permissions
- Docker installed and running
- Models trained (Section 3)
- SHAP/FFA analysis complete (Section 4)
- Risk distributions computed (if needed)

### Deployment Steps

1. **Prepare Lambda Directory**: Copy models, dashboard data, and risk distributions
2. **Build Docker Image**: Create container image with models
3. **Push to ECR**: Upload image to AWS Elastic Container Registry
4. **Update Lambda**: Deploy new container image to Lambda function
5. **Setup API Gateway**: Configure REST API endpoints
6. **Upload HTML**: Deploy dashboard HTML to S3

In [ ]:
# Step 1: Prepare Lambda Directory (Idempotent - only updates if needed)
import subprocess
from pathlib import Path
import os

print(f"\n{'=' * 80}")
print("Step 1: Preparing Lambda Directory")
print(f"{'=' * 80}")

risk_dashboard_dir = CALCULATOR_DIR / "risk_dashboard"
prepare_script = risk_dashboard_dir / "prepare_lambda_dir_phts.py"
lambda_dir = risk_dashboard_dir / "lambda_dir_phts"

# Check if Lambda directory already exists and is up to date
needs_prepare = True
if lambda_dir.exists():
    # Check if models exist and are recent
    models_dir = lambda_dir / "models" / COHORT
    dashboard_dir = lambda_dir / "dashboard_data" / COHORT
    
    # Check if source models are newer than lambda_dir models
    source_models_dir = CALCULATOR_DIR / "outputs" / "models" / COHORT
    source_dashboard_dir = CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT
    
    if models_dir.exists() and source_models_dir.exists():
        # Get most recent model file modification time
        source_model_files = list(source_models_dir.glob("*.cbm")) + list(source_models_dir.glob("*.ubj"))
        lambda_model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
        
        if source_model_files and lambda_model_files:
            source_mtime = max(f.stat().st_mtime for f in source_model_files)
            lambda_mtime = max(f.stat().st_mtime for f in lambda_model_files)
            
            if source_mtime <= lambda_mtime:
                print(f"\n✓ Lambda directory is up to date")
                print(f"  Source models: {len(source_model_files)} files")
                print(f"  Lambda models: {len(lambda_model_files)} files")
                print(f"  Last update: {lambda_mtime}")
                needs_prepare = False

if needs_prepare and prepare_script.exists():
    print(f"\nRunning: {prepare_script}")
    print("This will copy/update models, dashboard data, and risk distributions to lambda_dir_phts/")
    print("-" * 80)
    
    try:
        result = subprocess.run(
            [sys.executable, str(prepare_script)],
            cwd=str(risk_dashboard_dir),
            capture_output=False,
            text=True
        )
        
        if result.returncode == 0:
            print(f"\n✓ Lambda directory prepared successfully!")
            
            # Check what was created/updated
            if lambda_dir.exists():
                print(f"\n  Lambda directory structure:")
                print(f"    {lambda_dir}")
                
                # Check models
                models_dir = lambda_dir / "models" / COHORT
                if models_dir.exists():
                    model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
                    print(f"    ✓ Models: {len(model_files)} files")
                
                # Check dashboard data
                dashboard_dir = lambda_dir / "dashboard_data" / COHORT
                if dashboard_dir.exists():
                    dashboard_files = list(dashboard_dir.glob("*.json")) + list(dashboard_dir.glob("*.csv"))
                    print(f"    ✓ Dashboard data: {len(dashboard_files)} files")
                
                # Check risk distributions
                risk_dist_dir = lambda_dir / "risk_distributions"
                if risk_dist_dir.exists():
                    risk_files = list(risk_dist_dir.glob("*.json"))
                    print(f"    ✓ Risk distributions: {len(risk_files)} files")
        else:
            print(f"\n⚠ Script exited with code: {result.returncode}")
    except Exception as e:
        print(f"\n✗ Error preparing Lambda directory: {e}")
        logger.error("Error preparing Lambda directory", exc_info=True)
elif not prepare_script.exists():
    print(f"\n⚠ Prepare script not found: {prepare_script}")
    print("  Expected location: risk_dashboard/prepare_lambda_dir_phts.py")

print(f"\n{'=' * 80}")

In [ ]:
# Step 2: Build and Push Docker Image (Idempotent - only if Lambda dir changed)
print(f"\n{'=' * 80}")
print("Step 2: Build and Push Docker Image")
print(f"{'=' * 80}")

docker_script = risk_dashboard_dir / "docker_build_phts.sh"

# Check if Docker image needs to be rebuilt
# Rebuild if lambda_dir was just updated or if image doesn't exist
needs_docker_build = needs_prepare  # Rebuild if we just prepared lambda_dir

if docker_script.exists():
    print(f"\nDocker build strategy:")
    print(f"  - Lambda directory was {'updated' if needs_prepare else 'unchanged'}")
    print(f"  - Docker image will be {'built' if needs_docker_build else 'skipped (use --force to rebuild)'}")
    print("-" * 80)
    print("\nThis will:")
    print("  1. Build Docker image with models and dependencies")
    print("  2. Push image to AWS ECR (Elastic Container Registry)")
    print("-" * 80)
    print("\n⚠ Note: This requires:")
    print("  - Docker installed and running")
    print("  - AWS CLI configured with ECR permissions")
    print("  - AWS credentials with push access to ECR")
    print("-" * 80)
    
    if needs_docker_build:
        response = input("\nProceed with Docker build? (y/n): ").strip().lower()
    else:
        print("\n⏭ Skipping Docker build (Lambda directory unchanged)")
        print("  To force rebuild, run: ./docker_build_phts.sh --force")
        response = 'n'
    
    if response == 'y':
        try:
            result = subprocess.run(
                ["bash", str(docker_script)],
                cwd=str(risk_dashboard_dir),
                capture_output=False,
                text=True
            )
            
            if result.returncode == 0:
                print(f"\n✓ Docker image built and pushed successfully!")
                print(f"\n  Next: Get ECR URI from output above and use it to update Lambda")
            else:
                print(f"\n⚠ Docker build exited with code: {result.returncode}")
        except Exception as e:
            print(f"\n✗ Error building Docker image: {e}")
            logger.error("Error building Docker image", exc_info=True)
else:
    print(f"\n⚠ Docker build script not found: {docker_script}")
    print("  Expected location: risk_dashboard/docker_build_phts.sh")

print(f"\n{'=' * 80}")

In [ ]:
# Step 3: Update Lambda Function (Idempotent - only if Docker image was updated)
print(f"\n{'=' * 80}")
print("Step 3: Update Lambda Function")
print(f"{'=' * 80}")

lambda_function_name = "phts-risk-calculator"
region = "us-east-1"

# Check if Lambda function exists
lambda_exists = False
try:
    result = subprocess.run(
        ["aws", "lambda", "get-function", "--function-name", lambda_function_name, "--region", region],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        lambda_exists = True
        print(f"\n✓ Lambda function exists: {lambda_function_name}")
except:
    print(f"\n⚠ Could not check Lambda function status")
    print("  (AWS CLI may not be configured)")

if lambda_exists and needs_docker_build:
    print(f"\nLambda function will be updated with new Docker image")
    print("-" * 80)
    print("\n⚠ Note: AWS credentials are automatically used from EC2 instance role")
    print("  (No need to run 'aws configure' if instance has IAM role attached)")
    print("-" * 80)
    
    # Get AWS account ID and ECR URI
    try:
        result = subprocess.run(
            ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            account_id = result.stdout.strip()
            ecr_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/phts-risk-calculator:latest"
            print(f"\n  AWS Account ID: {account_id}")
            print(f"  ECR URI: {ecr_uri}")
            
            # Also get identity to show which role is being used
            identity_result = subprocess.run(
                ["aws", "sts", "get-caller-identity", "--output", "json"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if identity_result.returncode == 0:
                import json
                identity = json.loads(identity_result.stdout)
                if "Arn" in identity:
                    print(f"  Using IAM Role: {identity['Arn']}")
            
            response = input("\nProceed with Lambda update? (y/n): ").strip().lower()
            
            if response == 'y':
                try:
                    result = subprocess.run(
                        [
                            "aws", "lambda", "update-function-code",
                            "--function-name", lambda_function_name,
                            "--image-uri", ecr_uri,
                            "--region", region
                        ],
                        capture_output=False,
                        text=True
                    )
                    
                    if result.returncode == 0:
                        print(f"\n✓ Lambda function updated successfully!")
                        print(f"\n  Waiting for update to complete...")
                        # Wait for function to be ready
                        subprocess.run(
                            ["aws", "lambda", "wait", "function-updated",
                             "--function-name", lambda_function_name,
                             "--region", region],
                            capture_output=True
                        )
                        print(f"  ✓ Lambda function is ready")
                    else:
                        print(f"\n⚠ Lambda update exited with code: {result.returncode}")
                except Exception as e:
                    print(f"\n✗ Error updating Lambda: {e}")
                    logger.error("Error updating Lambda", exc_info=True)
            else:
                print("\n⏭ Skipping Lambda update")
        else:
            print("\n⚠ Could not retrieve AWS Account ID")
    except:
        print("\n⚠ Could not retrieve AWS Account ID - check AWS CLI configuration")
elif not lambda_exists:
    print(f"\n⚠ Lambda function '{lambda_function_name}' does not exist")
    print("  Create it first using AWS Console or CLI")
elif not needs_docker_build:
    print(f"\n⏭ Skipping Lambda update (Docker image unchanged)")
else:
    print("\nTo update Lambda function manually:")
    print("-" * 80)
    print("\n1. Get your AWS Account ID:")
    print("   AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)")
    print("\n2. Construct ECR URI:")
    print("   ECR_URI=\"${AWS_ACCOUNT_ID}.dkr.ecr.us-east-1.amazonaws.com/phts-risk-calculator:latest\"")
    print("\n3. Update Lambda function:")
    print("   aws lambda update-function-code \\")
    print("       --function-name phts-risk-calculator \\")
    print("       --image-uri ${ECR_URI} \\")
    print("       --region us-east-1")

print(f"\n{'=' * 80}")

In [ ]:
# Step 4: Verify API Gateway (Idempotent - assumes already set up)
print(f"\n{'=' * 80}")
print("Step 4: Verify API Gateway")
print(f"{'=' * 80}")

print("\n⚠ Note: API Gateway should already be set up")
print("  This step only verifies the configuration")
print("-" * 80)

# Check if API Gateway exists by trying to list APIs
try:
    result = subprocess.run(
        ["aws", "apigateway", "get-rest-apis", "--query", "items[?name=='phts-risk-calculator-api'].id", "--output", "text"],
        capture_output=True,
        text=True,
        timeout=5
    )
    
    if result.returncode == 0 and result.stdout.strip():
        api_id = result.stdout.strip()
        api_url = f"https://{api_id}.execute-api.us-east-1.amazonaws.com/prod"
        print(f"\n✓ API Gateway found:")
        print(f"  API ID: {api_id}")
        print(f"  API URL: {api_url}")
        
        # Test metadata endpoint
        print(f"\n  Testing /metadata endpoint...")
        try:
            test_result = subprocess.run(
                ["curl", "-s", f"{api_url}/metadata?cohort={COHORT}"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if test_result.returncode == 0:
                print(f"  ✓ API Gateway is responding")
            else:
                print(f"  ⚠ API Gateway may not be responding correctly")
        except:
            print(f"  ⚠ Could not test API endpoint (curl may not be available)")
    else:
        print(f"\n⚠ API Gateway not found or AWS CLI not configured")
        print("  If API Gateway needs to be set up, run:")
        print(f"    cd {risk_dashboard_dir}")
        print(f"    ./setup_api_gateway.sh")
except:
    print(f"\n⚠ Could not verify API Gateway (AWS CLI may not be configured)")
    print("  Assuming API Gateway is already set up")

print(f"\n{'=' * 80}")

In [ ]:
# Step 5: Upload HTML to S3 (Idempotent - only if HTML changed)
print(f"\n{'=' * 80}")
print("Step 5: Upload HTML Dashboard to S3")
print(f"{'=' * 80}")

html_file = risk_dashboard_dir / "phts_dashboard.html"
s3_bucket = "jerome-dixon.io"  # Update if different
s3_prefix = "uva/phts-risk-calculator"
s3_path = f"s3://{s3_bucket}/{s3_prefix}/index.html"

if html_file.exists():
    print(f"\nHTML file found: {html_file}")
    
    # Check if S3 file exists and compare modification times
    needs_upload = True
    try:
        result = subprocess.run(
            ["aws", "s3", "ls", s3_path, "--region", "us-east-1"],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0 and result.stdout.strip():
            # S3 file exists - check if local is newer
            local_mtime = html_file.stat().st_mtime
            
            # Parse S3 last modified time from ls output
            # Format: "2026-01-26 10:30:45    12345 index.html"
            s3_output = result.stdout.strip()
            if s3_output:
                print(f"\n✓ S3 file exists")
                print(f"  Checking if local file is newer...")
                
                # Get S3 file metadata
                head_result = subprocess.run(
                    ["aws", "s3api", "head-object", "--bucket", s3_bucket, 
                     "--key", f"{s3_prefix}/index.html", "--region", "us-east-1"],
                    capture_output=True,
                    text=True,
                    timeout=5
                )
                
                if head_result.returncode == 0:
                    import json
                    s3_meta = json.loads(head_result.stdout)
                    s3_mtime_str = s3_meta.get("LastModified", "")
                    if s3_mtime_str:
                        from datetime import datetime
                        s3_mtime = datetime.fromisoformat(s3_mtime_str.replace("Z", "+00:00")).timestamp()
                        
                        if local_mtime <= s3_mtime:
                            print(f"  ✓ S3 file is up to date (local: {local_mtime}, S3: {s3_mtime})")
                            needs_upload = False
                        else:
                            print(f"  ⚠ Local file is newer - will upload")
        else:
            print(f"\n⚠ S3 file not found - will upload")
    except:
        print(f"\n⚠ Could not check S3 file status (AWS CLI may not be configured)")
        print(f"  Will attempt upload")
    
    if needs_upload:
        print(f"\nUploading to S3:")
        print("-" * 80)
        print(f"  Source: {html_file}")
        print(f"  Destination: {s3_path}")
        print("-" * 80)
        print("\n⚠ Note: This requires:")
        print("  - AWS CLI configured")
        print("  - S3 write permissions")
        print("  - Bucket exists and is accessible")
        print("-" * 80)
        
        response = input("\nProceed with S3 upload? (y/n): ").strip().lower()
        
        if response == 'y':
            try:
                result = subprocess.run(
                    [
                        "aws", "s3", "cp",
                        str(html_file),
                        s3_path,
                        "--content-type", "text/html",
                        "--cache-control", "no-cache",
                        "--region", "us-east-1"
                    ],
                    capture_output=False,
                    text=True
                )
                
                if result.returncode == 0:
                    print(f"\n✓ HTML uploaded successfully to S3!")
                    print(f"\n  Dashboard URL: https://{s3_bucket}/{s3_prefix}/")
                else:
                    print(f"\n⚠ S3 upload exited with code: {result.returncode}")
            except Exception as e:
                print(f"\n✗ Error uploading to S3: {e}")
                logger.error("Error uploading to S3", exc_info=True)
        else:
            print("\n⏭ Skipping S3 upload")
    else:
        print(f"\n⏭ Skipping S3 upload (file is up to date)")
else:
    print(f"\n⚠ HTML file not found: {html_file}")
    print("  Expected location: risk_dashboard/phts_dashboard.html")

print(f"\n{'=' * 80}")

### Deployment Verification

After deployment, verify all components are working:

1. **Lambda Function**: Check CloudWatch logs
2. **API Gateway**: Test endpoints (`/metadata`, `/risk`, `/causal`)
3. **S3 Dashboard**: Load HTML page and test risk calculation
4. **CORS**: Verify browser can call API without CORS errors

### Quick Deployment Script

For automated deployment, use the complete deployment script:

```bash
cd graft-loss/cohort_analysis/calculator/risk_dashboard
./deploy_complete.sh
```

This script automates all deployment steps.

### Documentation

For detailed deployment instructions, see:
- `docs/calculator/README_deployment.md` - Complete deployment guide
- `risk_dashboard/README_DEPLOYMENT.md` - Deployment reference
- `risk_dashboard/README_ARCHITECTURE.md` - Architecture overview

In [ ]:
# Final Step: Shutdown EC2 Instance (Optional)
# Set SHUTDOWN_EC2 = True to enable, False to disable
SHUTDOWN_EC2 = False  # Change to True to enable auto-shutdown

print(f"\n{'=' * 80}")
print("Final Step: EC2 Instance Shutdown (Optional)")
print(f"{'=' * 80}")

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)
    
    import subprocess
    import shutil
    import os
    
    # Get instance ID from EC2 metadata service
    try:
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5
        )
        instance_id = result.stdout.strip()
        
        if instance_id and len(instance_id) > 0:
            print(f"Instance ID: {instance_id}")
            
            # Find AWS CLI
            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                # Try common paths
                aws_paths = [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws"
                ]
                for path in aws_paths:
                    if os.path.exists(path):
                        aws_cmd = path
                        break
            
            if aws_cmd:
                # Stop the instance (use terminate-instances for permanent deletion)
                shutdown_cmd = [aws_cmd, "ec2", "stop-instances", "--instance-ids", instance_id]
                
                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(shutdown_cmd, capture_output=True, text=True)
                
                if result.returncode == 0:
                    print("\n✓ EC2 instance stop command sent successfully")
                    print("Instance will stop in a few moments.")
                    print("Note: This is a STOP (not terminate), so you can restart it later.")
                    logger.info(f"EC2 instance {instance_id} stop command sent successfully")
                else:
                    print(f"\n⚠ EC2 stop command returned exit code {result.returncode}.")
                    print("Check AWS credentials and permissions.")
                    if result.stderr:
                        print(f"Error: {result.stderr}")
                    logger.warning(f"EC2 stop command failed: {result.stderr}")
            else:
                print("\nWarning: AWS CLI not found. Cannot shutdown instance.")
                print("Install AWS CLI or ensure it's in your PATH.")
                logger.warning("AWS CLI not found, cannot shutdown EC2 instance")
        else:
            print("\nWarning: Could not determine instance ID. Skipping shutdown.")
            print("If you want to shutdown manually, use:")
            print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
            logger.warning("Could not determine EC2 instance ID")
    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout retrieving instance ID from metadata service.")
        print("If running on EC2, check that metadata service is accessible.")
        logger.warning("Timeout retrieving EC2 instance ID from metadata service")
    except Exception as e:
        print(f"\nWarning: Could not retrieve instance ID: {e}")
        print("If you want to shutdown manually, use:")
        print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
        logger.warning(f"Error retrieving EC2 instance ID: {e}")
else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("To enable auto-shutdown, set SHUTDOWN_EC2 = True in this cell.")
    print("Instance will continue running.")

print(f"\n{'=' * 80}")
print("Workflow Complete!")
print(f"{'=' * 80}")

In [ ]:
# Validate features
print(f"\n{'=' * 80}")
print("Feature Validation")
print(f"{'=' * 80}")

try:
    from validate_features import validate_features
    
    print(f"\nValidating features for {COHORT} model...")
    validation_result = validate_features(COHORT)
    
    print(f"\nValidation Results:")
    print(f"  Status: {'✓ PASSED' if validation_result['status'] == 'passed' else '✗ FAILED'}")
    print(f"  Model features: {validation_result['model_feature_count']}")
    print(f"  Training features: {validation_result['training_feature_count']}")
    
    if validation_result['status'] == 'passed':
        print(f"\n  ✓ All features aligned!")
        print(f"  ✓ Model and training features match")
    else:
        print(f"\n  ⚠ Feature mismatch detected:")
        if validation_result.get('missing_in_model'):
            print(f"    Missing in model ({len(validation_result['missing_in_model'])}):")
            for feat in validation_result['missing_in_model'][:10]:
                print(f"      - {feat}")
        if validation_result.get('missing_in_training'):
            print(f"    Missing in training ({len(validation_result['missing_in_training'])}):")
            for feat in validation_result['missing_in_training'][:10]:
                print(f"      - {feat}")
    
    # Check required features
    print(f"\n  Required Features Check:")
    required_features = {
        'primary_etiology': 'primary_etiology',
        'hxsurg': 'hxsurg',
        'chd_lat': 'chd_lat',
        'hxdysdia': 'hxdysdia',
        'ecmo_combined': 'ecmo_combined',
        'vad_combined': 'vad_combined',
        'vent_combined': 'vent_combined',
        'age_txpl': 'age_txpl',
        'egfr_tx': 'egfr_tx',
        'txalt': 'txalt',
        'txast': 'txast',
        'txbili_d_r': 'txbili_d_r',
        'txbili_t_r': 'txbili_t_r',
        'txsa_r': 'txsa_r',
        'txtp_r': 'txtp_r',
        'txfcpra': 'txfcpra',
        'donisch': 'donisch',
        'donor_weight_ratio': 'donor_weight_ratio',
        'donor_size_ratio': 'donor_size_ratio'
    }
    
    model_features = set(validation_result.get('model_features', []))
    for req_name, req_var in required_features.items():
        # Check exact match or contains
        found = req_var in model_features or any(req_var in f for f in model_features)
        status = "✓" if found else "✗"
        print(f"    {status} {req_name:25s} ({req_var})")
    
except ImportError:
    print("\n⚠ validate_features module not found")
    print("  Run: python validate_features.py --cohort Combined")
except Exception as e:
    print(f"\n✗ Error validating features: {e}")
    logger.error("Error validating features", exc_info=True)

print(f"\n{'=' * 80}")